In [1]:
LEAGUE = "brasileirao-serie-a"
SEASON = "2024"
ENVIRONMENT = "prd"
REPROC_MODE = True

In [2]:
import boto3
import json
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("raw") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://cvmc-minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

s3 = boto3.client(
    's3',
    endpoint_url='http://cvmc-minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

print(f"Spark version: {spark.version}")
print("Delta + MinIO + S3 configurados!")

Spark version: 3.5.0
Delta + MinIO + S3 configurados!


In [3]:
# =====================
# PARÂMETROS DE EXECUÇÃO
# =====================
# LEAGUE = "brasileirao-serie-a"
# SEASON = "2026"
# ENVIRONMENT = "dev"
# REPROC_MODE = True

# =====================
# CONFIGURAÇÕES FIXAS
# =====================
BUCKET = "eng-prd-data-lake"

LANDING_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/landing/{LEAGUE}"
RAW_PATH = f"s3a://{BUCKET}/{ENVIRONMENT}/raw/{LEAGUE}"

print(f"League: {LEAGUE} | Season: {SEASON} | Env: {ENVIRONMENT} | Reproc: {REPROC_MODE}")
print(f"Landing: {LANDING_PATH}")
print(f"Raw:     {RAW_PATH}")

League: brasileirao-serie-a | Season: 2024 | Env: prd | Reproc: True
Landing: s3a://eng-prd-data-lake/prd/landing/brasileirao-serie-a
Raw:     s3a://eng-prd-data-lake/prd/raw/brasileirao-serie-a


In [4]:
from pyspark.sql.functions import current_timestamp, lit, explode, col
from pyspark.sql.types import StructType, ArrayType, StructField
import json

# === AS SUAS FUNÇÕES DE CONTROLE DE ARQUIVOS (MANTIDAS) ===
CHECKPOINT_KEY = f"{ENVIRONMENT}/raw/{LEAGUE}/_processed_files.json"

def get_processed_files():
    try:
        obj = s3.get_object(Bucket=BUCKET, Key=CHECKPOINT_KEY)
        return set(json.loads(obj['Body'].read()))
    except:
        return set()

def save_processed_files(files):
    s3.put_object(
        Bucket=BUCKET, Key=CHECKPOINT_KEY,
        Body=json.dumps(list(files)), ContentType="application/json"
    )

def list_landing_files(data_type):
    prefix = f"{ENVIRONMENT}/landing/{LEAGUE}/season={SEASON}/{data_type}/"
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix)
    return [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.json')]


# === A NOVA INGESTÃO BATCH À PROVA DE BALAS ===
def ingest_to_raw_batch(data_type):
    print(f"\n[INFO] Iniciando ingestão Batch para: {data_type} | REPROC_MODE: {REPROC_MODE}")
    
    landing_path = f"s3a://{BUCKET}/{ENVIRONMENT}/landing/{LEAGUE}/season={SEASON}/{data_type}/"
    raw_path = f"{RAW_PATH}/{data_type}"
    
    # 1. Lista todos os arquivos da Landing e pega o checkpoint atual
    all_landing_files = list_landing_files(data_type)
    processed = get_processed_files()

    # 2. Lógica do REPROC_MODE
    if REPROC_MODE:
        print("  -> [REPROC_MODE=True] Lendo TODOS os arquivos da Landing...")
        files_to_process = all_landing_files
        write_mode = "overwrite"
    else:
        files_to_process = [f for f in all_landing_files if f not in processed]
        write_mode = "append"

    if not files_to_process:
        print(f"  -> [SKIP] Nenhum arquivo para processar.")
        return

    print(f"  -> {len(files_to_process)} arquivo(s) na fila de processamento.")
    files_paths = [f"s3a://{BUCKET}/{f}" for f in files_to_process]

    # 3. Inferência de Schema (Mantida como conversamos)
    json_schema = None
    try:
        df_delta = spark.read.format("delta").load(raw_path)
        record_schema = next(f.dataType for f in df_delta.schema.fields if f.name == "record")
        json_schema = StructType([StructField("data", ArrayType(record_schema), True)])
    except Exception:
        print("  -> Tabela Delta não existe. Inferindo schema da Landing...")
        full_landing_df = spark.read.option("multiline", "true").json(landing_path)
        json_schema = full_landing_df.schema

    # 4. Leitura e Transformação
    df_lote = spark.read.option("multiline", "true").schema(json_schema).json(files_paths)

    df_clean = df_lote.withColumn("record", explode(col("data"))) \
                       .drop("data") \
                       .withColumn("ingested_at", current_timestamp()) \
                       .withColumn("league", lit(LEAGUE)) \
                       .withColumn("season", lit(SEASON))

    # 5. Escrita inteligente no Delta Lake
    print(f"  -> Salvando dados (Modo: {write_mode})...")
    writer = df_clean.write.format("delta") \
                     .mode(write_mode) \
                     .option("mergeSchema", "true") \
                     .partitionBy("season")
    
    # O Pulo do Gato: Protege as outras temporadas se for overwrite
    if write_mode == "overwrite":
        writer = writer.option("replaceWhere", f"season = '{SEASON}'")
        
    writer.save(raw_path)

    # 6. Atualiza o state no S3 preservando o histórico de outras seasons e data_types
    if REPROC_MODE:
        # Adiciona TODOS os arquivos desta season/data_type ao set existente
        processed.update(all_landing_files)
    else:
        # Adiciona APENAS os arquivos novos ao set existente
        processed.update(files_to_process)
            
    save_processed_files(processed)
    
    print(f"  -> [OK] {data_type} processado com sucesso!")


# Executa o loop
for data_type in ["standing", "rounds", "events", "players", "team_stats", "player_stats"]:
    print(data_type)
    ingest_to_raw_batch(data_type)

print("\nProcesso da camada Raw finalizado!")

standing

[INFO] Iniciando ingestão Batch para: standing | REPROC_MODE: True
  -> [REPROC_MODE=True] Lendo TODOS os arquivos da Landing...
  -> 2 arquivo(s) na fila de processamento.
  -> Salvando dados (Modo: overwrite)...
  -> [OK] standing processado com sucesso!
rounds

[INFO] Iniciando ingestão Batch para: rounds | REPROC_MODE: True
  -> [REPROC_MODE=True] Lendo TODOS os arquivos da Landing...
  -> 2 arquivo(s) na fila de processamento.
  -> Salvando dados (Modo: overwrite)...
  -> [OK] rounds processado com sucesso!
events

[INFO] Iniciando ingestão Batch para: events | REPROC_MODE: True
  -> [REPROC_MODE=True] Lendo TODOS os arquivos da Landing...
  -> 2 arquivo(s) na fila de processamento.
  -> Salvando dados (Modo: overwrite)...
  -> [OK] events processado com sucesso!
players

[INFO] Iniciando ingestão Batch para: players | REPROC_MODE: True
  -> [REPROC_MODE=True] Lendo TODOS os arquivos da Landing...
  -> 2 arquivo(s) na fila de processamento.
  -> Salvando dados (Modo: ove